# Лабораторная работа № 4. Гипотезы и статистические тесты

В работе исследуются данные пренатальной диагностики, психологов и цветов Iris. Уровень значимости во всех проверках — **0,05**.

Перед запуском положите Excel-файлы рядом с ноутбуком либо оставьте их в папке `Загрузки`.

In [ ]:
import sys

# Установка библиотеки для Excel при запуске в JupyterLite
if sys.platform == "emscripten":
    import micropip
    await micropip.install("openpyxl")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path

# Файл ищется рядом с ноутбуком, затем в папке Загрузки
def nayti_fayl(imya):
    if Path(imya).exists():
        return imya
    return Path.home() / "Downloads" / imya

alfa = 0.05

## 1. Пренатальная диагностика

In [ ]:
dannye = pd.read_excel(nayti_fayl("Пренатальная диагностика.xlsx"), sheet_name="исходные данные")
dannye.head()

In [ ]:
print("Размер таблицы:", dannye.shape)
print("Пропуски:")
print(dannye.isna().sum())
print("Количество мальчиков и девочек:")
print(dannye["pol(1-м, 0-д)"].value_counts().rename(index={1: "Мальчики", 0: "Девочки"}))

In [ ]:
# ИМТ = вес в килограммах / рост в метрах в квадрате
dannye["ИМТ"] = dannye["ves"] / 1000 / (dannye["rost"] / 100) ** 2
chislovye_stolbtsy = ["vozrast", "gr-dni", "ctr", "nt", "rost", "ves", "ИМТ"]

opisanie = dannye[chislovye_stolbtsy].describe().T
opisanie["размах"] = opisanie["max"] - opisanie["min"]
opisanie["медиана"] = dannye[chislovye_stolbtsy].median()
opisanie[["count", "min", "max", "размах", "mean", "медиана", "std"]].round(2)

Для проверки нормальности используем критерий Шапиро—Уилка. H₀: распределение нормальное; H₁: распределение отличается от нормального. При p-value < 0,05 отвергаем H₀.

In [ ]:
for stolbets in chislovye_stolbtsy:
    znacheniya = dannye[stolbets].dropna()
    statistika, p = stats.shapiro(znacheniya)
    print("\nПоказатель:", stolbets)
    print("Асимметрия:", round(stats.skew(znacheniya), 3))
    print("Эксцесс:", round(stats.kurtosis(znacheniya), 3))
    print("p-value:", round(p, 4))
    if p < alfa:
        print("Распределение отличается от нормального")
    else:
        print("Нет оснований считать распределение ненормальным")
    plt.hist(znacheniya, bins=10, edgecolor="black")
    plt.title(stolbets)
    plt.show()

### Сравнение мальчиков и девочек

H₀: средние значения у мальчиков и девочек равны. H₁: средние отличаются. Сравниваем две независимые группы двусторонним t-тестом Уэлча: он подходит, даже если дисперсии групп неодинаковы.

In [ ]:
malchiki = dannye[dannye["pol(1-м, 0-д)"] == 1]
devochki = dannye[dannye["pol(1-м, 0-д)"] == 0]

for stolbets in ["rost", "ves", "ИМТ"]:
    t, p = stats.ttest_ind(malchiki[stolbets], devochki[stolbets], equal_var=False)
    print("\nПоказатель:", stolbets)
    print("Среднее у мальчиков:", round(malchiki[stolbets].mean(), 2))
    print("Среднее у девочек:", round(devochki[stolbets].mean(), 2))
    print("t-статистика:", round(t, 3), "p-value:", round(p, 4))
    if p < alfa:
        print("Средние статистически различаются")
    else:
        print("Статистически значимого различия не найдено")

## 2. Т-тесты по данным психологов

In [ ]:
psihologi = pd.read_excel(nayti_fayl("Психологи.xlsx"))
psihologi.columns = psihologi.columns.str.strip()
psihologi["пол"] = psihologi["пол"].astype(str).str.strip().str.lower()
psihologi.head()

H₀ для каждого показателя: средние значения у девушек и юношей равны. H₁: средние отличаются. Используем двусторонний t-тест Уэлча для независимых групп.

In [ ]:
devushki = psihologi[psihologi["пол"] == "ж"]
yunoshi = psihologi[psihologi["пол"] == "м"]

# Убираем номер ученика и категориальные признаки; остальные числа — результаты тестов
pokazateli = [stolbets for stolbets in psihologi.select_dtypes(include="number").columns
              if stolbets not in ["учащиеся", "класс", "профиль ВУЗа"]]

rezultaty = []
for stolbets in pokazateli:
    t, p = stats.ttest_ind(devushki[stolbets].dropna(), yunoshi[stolbets].dropna(), equal_var=False)
    rezultaty.append([stolbets, devushki[stolbets].mean(), yunoshi[stolbets].mean(), t, p])

tablitsa_testov = pd.DataFrame(rezultaty, columns=["Показатель", "Среднее у девушек", "Среднее у юношей", "t", "p-value"])
tablitsa_testov["Вывод"] = np.where(tablitsa_testov["p-value"] < alfa,
                                      "Различие есть", "Значимого различия нет")
tablitsa_testov.sort_values("p-value").round(3)

In [ ]:
# Сравним размер различий по абсолютной разнице средних
tablitsa_testov["Разница средних"] = abs(tablitsa_testov["Среднее у девушек"] - tablitsa_testov["Среднее у юношей"])
print("Самая большая разница:", tablitsa_testov.loc[tablitsa_testov["Разница средних"].idxmax(), "Показатель"])
print("Самая маленькая разница:", tablitsa_testov.loc[tablitsa_testov["Разница средних"].idxmin(), "Показатель"])

## 3. Бонус. Проверка нормальности Iris

Для критерия χ² Пирсона разобьём данные на 8 интервалов с равными вероятностями по нормальному распределению. Среднее и стандартное отклонение берём из выборки. H₀: данные имеют нормальное распределение; H₁: распределение ненормальное.

In [ ]:
iris_sort = pd.read_excel(nayti_fayl("Iris.xlsx"), sheet_name="Data")
vidy = iris_sort["Species_name"]
tsvety = iris_sort.drop(columns=["Species_No", "Species_name"])
tsvety.columns = ["Ширина лепестка", "Длина лепестка", "Ширина чашелистика", "Длина чашелистика"]
tsvety.index = range(1, len(tsvety) + 1)
tsvety.head()

In [ ]:
def gauss(dannye, nazvanie):
    znacheniya = dannye.dropna().to_numpy()
    n = len(znacheniya)
    srednee = znacheniya.mean()
    otklonenie = znacheniya.std(ddof=1)

    # Границы 8 интервалов берём из нормального распределения
    granitsy = stats.norm.ppf(np.linspace(0, 1, 9), loc=srednee, scale=otklonenie)
    granitsy[0], granitsy[-1] = -np.inf, np.inf
    nablyudaemye = np.histogram(znacheniya, bins=granitsy)[0]
    ozhidaemye = np.full(8, n / 8)

    hi_kvadrat = sum((nablyudaemye - ozhidaemye) ** 2 / ozhidaemye)
    stepeni_svobody = 8 - 1 - 2  # оценивали среднее и стандартное отклонение
    p = stats.chi2.sf(hi_kvadrat, stepeni_svobody)

    print(nazvanie)
    print("Объём выборки:", n)
    print("Среднее:", round(srednee, 3), "Медиана:", round(np.median(znacheniya), 3))
    print("Асимметрия:", round(stats.skew(znacheniya), 3))
    print("Эксцесс:", round(stats.kurtosis(znacheniya), 3))
    print("H₀: распределение нормальное; H₁: распределение ненормальное")
    print("χ²:", round(hi_kvadrat, 3), "p-value:", round(p, 4))
    if p < alfa:
        print("Нормальность отвергается")
    else:
        print("Нет оснований отвергнуть нормальность")

    plt.hist(znacheniya, bins=10, edgecolor="black")
    plt.title(nazvanie)
    plt.show()
    return p

In [ ]:
p_values = {}
for stolbets in tsvety.columns:
    p_values[stolbets] = gauss(tsvety[stolbets], stolbets)

samyy_nenormalnyy = min(p_values, key=p_values.get)
print("Самый ненормальный показатель:", samyy_nenormalnyy)

In [ ]:
# В исходной таблице виды идут группами по 50 цветков
for nomer, vid in enumerate(["Setosa", "Versicolor", "Virginica"]):
    nachalo = nomer * 50
    konets = nachalo + 50
    gauss(tsvety.iloc[nachalo:konets][samyy_nenormalnyy], vid)

In [ ]:
# Setosa — мелкие цветки; два других вида объединяем как крупные
gauss(tsvety.iloc[:50][samyy_nenormalnyy], "Мелкие цветки (Setosa)")
gauss(tsvety.iloc[50:][samyy_nenormalnyy], "Крупные цветки (Versicolor и Virginica)")

## Общий вывод

В таблицах выше приведены результаты всех проверок. Если p-value меньше 0,05, нулевая гипотеза отвергается. Для χ² вывод зависит от принятого разбиения на интервалы. Статистический тест показывает наличие различий в данных, но не объясняет их причину.